# 19. Run or resume the paired generalization study

The existing paired_generalization_v2 study is already prepared in R2.
Change only SEED in section 2 and run all cells on A100. Training is enabled by default.
Reopening with the same seed resumes verified unfinished work and skips completed jobs.
A new seed trains its own source history. Do not rerun PREPARE for this existing study.

Learning saves after each epoch. Unlearning saves at the configured step interval.
A runtime loss can discard work since the last verified R2 boundary. An interrupted
learning epoch restarts from its preceding boundary with the saved Adam, protection
snapshot, random state, and data-order generator. It does not start with a fresh optimizer.

The time budget is a soft limit: a boundary and its uploads may finish after the limit.
Use a generous reserve. A GPU/software mismatch stops resume rather than silently changing
the experiment. Resume support does not establish repeatability across different GPUs.

## 1. Load the fixed code

In [ ]:
from pathlib import Path
import sys, os, json, subprocess, importlib.util
IN_COLAB = "google.colab" in sys.modules
CODE_VERSION = "65e2e7c850fd6f1852a99d9e81780c27f799c926"
if any(name == "uncle" or name.startswith("uncle.") for name in sys.modules):
    raise RuntimeError("Restart the runtime before loading this fixed code revision.")
if IN_COLAB:
    REPO = Path("/content/cark-paired-study")
    if not REPO.exists():
        subprocess.run(["git", "clone", "https://github.com/sumitasthana/CARK.git", str(REPO)], check=True)
    if subprocess.check_output(["git", "-C", str(REPO), "status", "--porcelain", "--untracked-files=no"], text=True).strip():
        raise RuntimeError("Checkout has edits. Use a fresh runtime.")
    subprocess.run(["git", "-C", str(REPO), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", CODE_VERSION], check=True)
else:
    REPO = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "uncle/task_partition.json").exists()), None)
    if REPO is None:
        raise FileNotFoundError("Open this notebook from a CARK checkout.")
    actual = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
    implementation = ["uncle", "pyproject.toml", "requirements.txt"]
    dirty = subprocess.check_output(["git", "-C", str(REPO), "status", "--porcelain", "--", *implementation], text=True).strip()
    # A later documentation-only commit may contain the same fixed implementation.
    differences = subprocess.check_output(["git", "-C", str(REPO), "diff", CODE_VERSION, actual, "--", *implementation], text=True).strip()
    if dirty or differences:
        raise RuntimeError("Use a clean checkout of CODE_VERSION, or open in Colab. Local training edits are preserved.")
for package in ("boto3", "tqdm", "matplotlib"):
    if importlib.util.find_spec(package) is None:
        if not IN_COLAB:
            raise ImportError("Install the storage and extras dependencies before running this notebook.")
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", package], check=True)
sys.path.insert(0, str(REPO))
from uncle.storage import R2Store
from uncle.paired_study import StudyStore, make_plan, jobs, inspect_queue, run_queue, review
import torch
print("Fixed implementation revision:", CODE_VERSION)
print("PyTorch:", torch.__version__, "CUDA build:", torch.version.cuda)


## 2. Set the session budget

In [ ]:
STUDY_ID = "paired_generalization_v2"
BUCKET = "uncle-experiments"
SEED = 0  # Change only this value: 0, 1, or 2. Then run all cells.
if type(SEED) is not int or SEED not in (0, 1, 2):
    raise ValueError("Choose SEED = 0, 1, or 2 from the saved study.")
RUN_TRAINING = True
MAX_JOBS = 4
SELECTED_JOB_IDS = [
    f"sources/order_01/seed_{SEED}",
    f"controls/order_01/seed_{SEED}/learn_15",
    f"branches/order_01/seed_{SEED}/forget_3/learn_15",
    f"branches/order_01/seed_{SEED}/forget_14/learn_15",
]
MAX_SESSION_MINUTES = 120
SAVE_RESERVE_MINUTES = 15
CLEAN_COMPLETED_RESUME_SLOTS = True
RELEASE_GPU_WHEN_DONE = True
DATA_ROOT = Path("/content/data/tiny-imagenet-200") if IN_COLAB else REPO / "data/tiny-imagenet-200"
if not RUN_TRAINING:
    raise RuntimeError("Training is off. Set RUN_TRAINING = True when ready to run this session.")
if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime for this notebook.")
# These flags are recorded and must match on every resumed session.
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = False
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
print("GPU:", torch.cuda.get_device_name(0))
print("Session limit:", MAX_SESSION_MINUTES, "minutes; reserve:", SAVE_RESERVE_MINUTES)


## 3. Connect and recover the queue

In [ ]:
"""Notebook display only. Does not change training or persisted study state."""
import html
import time


class CompactBar:
    def __init__(self, dashboard, total, description, unit='step'):
        self.dashboard, self.total, self.description = dashboard, total or 0, description
        self.unit, self.n, self.detail = unit, 0, ''
        dashboard.bars.append(self)
        dashboard.render(force=True)

    def update(self, n=1):
        self.n += n
        self.dashboard.render()

    def set_postfix_str(self, text):
        self.detail = text
        self.dashboard.render()

    def reset(self, total=None):
        self.n = 0
        if total is not None:
            self.total = total
        self.dashboard.render(force=True)

    def write(self, text):
        self.dashboard.message = str(text)
        self.dashboard.render(force=True)

    def close(self):
        if self in self.dashboard.bars:
            self.dashboard.bars.remove(self)
        self.dashboard.render(force=True)

    def __enter__(self):
        return self

    def __exit__(self, *args):
        self.close()


class StudyDashboard:
    def __init__(self, seed, emit=None, clock=time.monotonic):
        self.seed, self.clock = seed, clock
        self.emit = emit
        self.handle = None
        self.last_render = float('-inf')
        self.bars, self.patches = [], []
        self.jobs, self.pointers = {}, {}
        self.current_job, self.current_request = None, ''
        self.message = 'Checking saved R2 progress'
        self.latest_save = 'No new save this session'

    def bar(self, total, description, enabled=True, leave=True):
        return CompactBar(self, total, description)

    def transfer_bar(self, total, description, enabled=True):
        return CompactBar(self, total, description, 'B')

    def download_bar(self, *args, **kwargs):
        return CompactBar(self, kwargs.get('total', 0),
            kwargs.get('desc') or 'Download dataset', kwargs.get('unit', 'B'))

    def install(self):
        import uncle.paired_study as paired
        import uncle.study_training as training
        import uncle.storage as storage_module
        import torchvision.datasets.utils as dataset_utils
        for module, attribute, replacement in (
            (paired, 'progress_bar', self.bar),
            (training, 'progress_bar', self.bar),
            (storage_module, '_transfer_bar', self.transfer_bar),
            (dataset_utils, 'tqdm', self.download_bar),
        ):
            self.patches.append((module, attribute, getattr(module, attribute)))
            setattr(module, attribute, replacement)
        self.render(force=True)

    def restore(self):
        for module, attribute, original in reversed(self.patches):
            setattr(module, attribute, original)
        self.patches.clear()

    def attach_here(self):
        # Keep the live panel beside the running cell, with no stale setup panel.
        if self.handle is not None:
            from IPython.display import HTML
            self.handle.update(HTML(''))
            self.handle = None
        self.render(force=True)

    def configure(self, plan, storage, all_jobs):
        # Count the narrowed study across all planned seeds, not the deferred 45-job matrix.
        self.jobs = {job['id']: job for job in all_jobs if job['order'] == 'order_01'
            and (job['kind'] != 'unlearned' or job.get('forget_task') in ('3', '14'))}
        self.pointers = {key: storage.progress(job) for key, job in self.jobs.items()}
        self.message = 'Ready; completed work will be skipped'
        self.render(force=True)

    def start_job(self, job):
        self.current_job = job['id']
        self.current_request = 'Restore model and prepare data'
        self.message = 'Running'
        self.render(force=True)

    def request(self, action, task):
        self.current_request = ('Learn ' if action == 'learn' else 'Unlearn ') + str(task)
        self.render(force=True)

    def saved(self, job, pointer):
        self.pointers[job['id']] = pointer
        self.latest_save = pointer['updated_at'] + ' (UTC)'
        self.render(force=True)

    def completed_requests(self, key):
        pointer = self.pointers.get(key)
        if not pointer:
            return 0
        job = self.jobs[key]
        total = len(job['operations'])
        if pointer['status'] == 'complete':
            return total
        inherited = len(self.jobs[job['source']]['operations']) if job['source'] else 0
        return max(0, min(total, pointer['completed_requests'] - inherited))

    def counts(self, seed=None):
        keys = [key for key, job in self.jobs.items() if seed is None or job['seed'] == seed]
        total = sum(len(self.jobs[key]['operations']) for key in keys)
        done = sum(self.completed_requests(key) for key in keys)
        complete = sum(bool(self.pointers.get(key) and
            self.pointers[key]['status'] == 'complete') for key in keys)
        pairs = sum(bool(self.pointers.get(key) and self.pointers[key]['status'] == 'complete'
            and (self.pointers.get('controls/' + self.jobs[key]['order'] + '/seed_' +
                str(self.jobs[key]['seed']) + '/learn_15') or {}).get('status') == 'complete')
            for key in keys if self.jobs[key]['kind'] == 'unlearned')
        return done, total, complete, len(keys), pairs

    @staticmethod
    def meter(n, total):
        return '<progress value="%s" max="%s" style="width:100%%"></progress>' % (n, total or 1)

    def markup(self):
        escape = lambda value: html.escape(str(value))
        done, total, complete, jobs_total, pairs = self.counts()
        seed_done, seed_total, seed_jobs, seed_jobs_total, seed_pairs = self.counts(self.seed)
        lines = ['<div style="border:1px solid #bbb;padding:12px;max-width:850px">',
            '<b>Paired study: order 01, U3/L15 and U14/L15</b>',
            '<p>Source: L3 → L0 → L9 → L5 → L17 → L1 → L7 → L14</p>',
            '<p>All planned seeds: %s/%s jobs complete; %s/%s comparisons complete; '
            '%s/%s requests saved complete.</p>' %
            (complete, jobs_total, pairs, jobs_total // 2, done, total),
            self.meter(done, total),
            '<p><b>Current seed %s:</b> %s/%s jobs; %s/2 comparisons; %s/%s requests saved complete.</p>' %
            (self.seed, seed_jobs, seed_jobs_total, seed_pairs, seed_done, seed_total),
            self.meter(seed_done, seed_total),
            '<p>Current job: %s<br>Request: %s<br>Status: %s</p>' %
            (escape(self.current_job or 'Not started'), escape(self.current_request), escape(self.message))]
        lines.append('<table><tr><th>Selected seed jobs</th><th>Saved status</th></tr>')
        for key, job in self.jobs.items():
            if job['seed'] != self.seed:
                continue
            pointer = self.pointers.get(key)
            label = ('Source (8 lessons)' if job['kind'] == 'source' else
                'Control: L15' if job['kind'] == 'control' else 'U%s then L15' % job['forget_task'])
            status = pointer['status'] if pointer else 'pending'
            lines.append('<tr><td>%s</td><td>%s</td></tr>' % (escape(label), escape(status)))
        lines.append('</table>')
        if self.bars:
            bar = self.bars[-1]
            amount = ('%.1f / %.1f MiB' % (bar.n / 2**20, bar.total / 2**20)
                if bar.unit == 'B' else '%s / %s steps' % (bar.n, bar.total))
            lines += ['<p>%s: %s. %s</p>' % (escape(bar.description), escape(amount), escape(bar.detail)),
                self.meter(bar.n, bar.total)]
        lines += ['<p>Latest verified R2 save: %s</p>' % escape(self.latest_save),
            '<small>Request counts are not GPU-time estimates. Deferred orders and targets are excluded.</small></div>']
        return ''.join(lines)

    def render(self, force=False):
        now = self.clock()
        if not force and now - self.last_render < 1.0:
            return
        self.last_render = now
        markup = self.markup()
        if self.emit:
            self.emit(markup)
            return
        from IPython.display import HTML, display
        if self.handle is None:
            self.handle = display(HTML(markup), display_id=True)
        else:
            self.handle.update(HTML(markup))

    def finish(self, message):
        self.message = message
        self.bars.clear()
        self.render(force=True)


dashboard = StudyDashboard(SEED)
dashboard.install()


In [ ]:
try:
    def secret(name):
        value = os.environ.get(name)
        if not value and IN_COLAB:
            from google.colab import userdata
            value = userdata.get(name)
        if not value:
            raise RuntimeError("Missing secret: " + name)
        return value
    store = R2Store.connect(secret("R2_ENDPOINT"), BUCKET,
        secret("R2_ACCESS_KEY_ID"), secret("R2_SECRET_ACCESS_KEY"))
    store.progress = True
    CACHE = REPO / "outputs/paired_study" / STUDY_ID
    CACHE.mkdir(parents=True, exist_ok=True)
    print("Connected to R2 bucket:", BUCKET)
    # Transport-only fix: preserve the saved training revision and resume contract.
    from boto3.s3.transfer import TransferConfig
    from boto3.exceptions import S3UploadFailedError
    from botocore.exceptions import ClientError
    import time
    transfer_config = TransferConfig(multipart_threshold=64 * 1024 * 1024,
        multipart_chunksize=64 * 1024 * 1024, max_concurrency=1,
        use_threads=False, preferred_transfer_client="classic")
    original_upload_file = store.client.upload_file
    def upload_with_multipart_retry(*args, **kwargs):
        kwargs["Config"] = transfer_config
        for attempt in range(1, 4):
            try:
                return original_upload_file(*args, **kwargs)
            except (S3UploadFailedError, ClientError) as error:
                error_code = getattr(error, "response", {}).get("Error", {}).get("Code")
                invalid_part = error_code == "InvalidPart" or (
                    isinstance(error, S3UploadFailedError) and "(InvalidPart)" in str(error))
                if not invalid_part or attempt == 3:
                    raise
                print("R2 InvalidPart: restarting the full upload, attempt", attempt + 1,
                    "of 3. Saved progress has not advanced.", flush=True)
                callback_owner = getattr(kwargs.get("Callback"), "__self__", None)
                if callback_owner is not None and hasattr(callback_owner, "reset"):
                    callback_owner.reset()
                time.sleep(2 * attempt)
    store.client.upload_file = upload_with_multipart_retry
    print("R2 uploads: sequential 64 MiB parts; at most three InvalidPart attempts.")
except BaseException:
    import traceback
    traceback.print_exc(file=sys.stdout)
    sys.stdout.flush()
    if "dashboard" in globals():
        dashboard.finish("Setup failed; see traceback")
        dashboard.restore()
    if RELEASE_GPU_WHEN_DONE and IN_COLAB:
        from google.colab import runtime
        print("Setup failed before training. Releasing the GPU runtime.")
        runtime.unassign()
    raise


In [ ]:
try:
    PREFIX = "uncle/paired_generalization/" + STUDY_ID
    path = store.download(PREFIX + "/study.json", CACHE / "cache")
    plan = json.loads(path.read_text())
    if plan["code_version"] != CODE_VERSION:
        raise ValueError("This study requires a different fixed code revision. Use its recorded notebook revision.")
    from uncle.learning_diagnostic import file_hash
    if file_hash(REPO / "uncle/task_partition.json") != plan["partition_sha256"]:
        raise ValueError("Class partition differs from the saved study.")
    storage = StudyStore(store, plan, CACHE, cleanup_completed=CLEAN_COMPLETED_RESUME_SLOTS)
    rows = inspect_queue(plan, storage, progress=True)
    if globals().get("SELECTED_JOB_IDS"):
        rows = [row for row in rows if row["id"] in SELECTED_JOB_IDS]
        print("Showing selected jobs only; the full saved study remains unchanged.")
    from collections import Counter
    print("Queue:", dict(Counter(row["status"] for row in rows)))
    for row in rows:
        print(row["status"].ljust(10), row["id"], row["active"] or "")
except BaseException:
    import traceback
    traceback.print_exc(file=sys.stdout)
    sys.stdout.flush()
    if "dashboard" in globals():
        dashboard.finish("Setup failed; see traceback")
        dashboard.restore()
    if RELEASE_GPU_WHEN_DONE and IN_COLAB:
        from google.colab import runtime
        print("Setup failed before training. Releasing the GPU runtime.")
        runtime.unassign()
    raise


In [ ]:
try:
    # Select execution only. The immutable plan and job contracts stay unchanged.
    import types
    from uncle.paired_study import inspect_queue as full_inspect_queue
    def selected_inspect_queue(saved_plan, saved_storage, *, progress=True):
        all_jobs = jobs(saved_plan)
        known = {job["id"]: job for job in all_jobs}
        if not SELECTED_JOB_IDS or len(set(SELECTED_JOB_IDS)) != len(SELECTED_JOB_IDS):
            raise ValueError("Select a nonempty list of distinct job IDs.")
        if set(SELECTED_JOB_IDS) - known.keys():
            raise ValueError("Selected job is outside the saved study.")
        selected = set(SELECTED_JOB_IDS)
        if any(known[key]["source"] and known[key]["source"] not in selected
               for key in selected):
            raise ValueError("Include the source job for each selected continuation.")
        rows = full_inspect_queue(saved_plan, saved_storage, progress=progress)
        by_id = {row["id"]: row for row in rows}
        return [by_id[key] for key in SELECTED_JOB_IDS]
    # Give only the session scheduler a filtered queue, without changing module globals.
    scheduler_globals = dict(run_queue.__globals__, inspect_queue=selected_inspect_queue)
    if "dashboard" in globals():
        dashboard.configure(plan, storage, jobs(plan))
        original_run_job = run_queue.__globals__["run_job"]
        original_request = original_run_job.__globals__["run_request"]
        def display_request(trainer, action, task, *args, **kwargs):
            dashboard.request(action, task)
            return original_request(trainer, action, task, *args, **kwargs)
        job_globals = dict(original_run_job.__globals__, run_request=display_request)
        display_job = types.FunctionType(original_run_job.__code__, job_globals,
            "display_job", original_run_job.__defaults__, original_run_job.__closure__)
        display_job.__kwdefaults__ = dict(original_run_job.__kwdefaults__ or {})
        def run_display_job(saved_plan, job, *args, **kwargs):
            dashboard.start_job(job)
            kwargs["on_boundary"] = dashboard.saved
            return display_job(saved_plan, job, *args, **kwargs)
        scheduler_globals["run_job"] = run_display_job
    run_selected_queue = types.FunctionType(run_queue.__code__, scheduler_globals,
        "run_selected_queue", run_queue.__defaults__, run_queue.__closure__)
    run_selected_queue.__kwdefaults__ = dict(run_queue.__kwdefaults__ or {})
    selected_rows = selected_inspect_queue(plan, storage, progress=True)
    print("Selected deadline queue:")
    for row in selected_rows:
        print(row["status"].ljust(10), row["id"], row["active"] or "")
    print("All other study jobs remain saved but will not run in this session.")
except BaseException:
    import traceback
    traceback.print_exc(file=sys.stdout)
    sys.stdout.flush()
    if "dashboard" in globals():
        dashboard.finish("Setup failed; see traceback")
        dashboard.restore()
    if RELEASE_GPU_WHEN_DONE and IN_COLAB:
        from google.colab import runtime
        print("Setup failed before training. Releasing the GPU runtime.")
        runtime.unassign()
    raise


## 4. Run the next work units

Change only SEED in section 2, then run all cells on A100. This queue uses
order 01 and the selected seed: source, L15-only control, U3 then L15,
followed by U14 then L15. Completed jobs are reused. Other orders and forget
targets are deferred. Keep the same study ID and saved plan.
MAX_JOBS = 4 allows all four jobs if the session budget permits; an unfinished
job stops the session and resumes next time with the same seed. A new seed
needs its own eight-lesson source. One session may not finish it.

Tiny ImageNet is downloaded to local runtime storage only when needed, with a download
progress bar. It is reused across the session. Evaluation, learning, unlearning, R2 transfers,
and the study queue also have progress bars.

Every progress pointer is published after its checkpoint and report are verified.
Source models remain available for every branch. By default, branch final models are
deleted after the completed report and model verification receipt are durably saved.
Immediate post-unlearning models are not retained by default. All reports, settings,
hashes, and figures remain. Unfinished jobs retain resumable model and optimizer state.
Cleanup touches only this study's completed-job model files and temporary slots.
It never deletes historical experiment files.

In [ ]:
from uncle.streams import build_tasks
dashboard.attach_here()
task_cache = {}
def tasks_factory(job):
    dashboard.start_job(job)
    wanted = tuple(sorted(set(plan["orders"][job["order"]]) |
        ({job["new_task"]} if job["source"] else set())))
    if wanted not in task_cache:
        # Dataset preparation occurs inside the timed session, not before its budget.
        task_cache[wanted] = build_tasks(root=DATA_ROOT,
            partition_path=REPO / "uncle/task_partition.json", partition_seed=42,
            include=wanted, download=True)
    return task_cache[wanted]

session = None
try:
    session = run_selected_queue(plan, storage, tasks_factory, max_jobs=MAX_JOBS,
        max_minutes=MAX_SESSION_MINUTES, reserve_minutes=SAVE_RESERVE_MINUTES,
        device="cuda", progress=True)
    print(json.dumps(session, indent=2))
    print("Session progress saved. Reopen this notebook with the same STUDY_ID to continue.")
except BaseException as error:
    import traceback
    traceback.print_exc(file=sys.stdout)
    sys.stdout.flush()
    print("Session interrupted:", type(error).__name__)
    print("Verified earlier R2 boundaries remain available. Do not change the study settings to resume.")
    raise
finally:
    dashboard.finish("Session saved; run all again to continue" if session is not None
        else "Interrupted; resume from the latest verified R2 save")
    dashboard.restore()
    task_cache.clear()
    import gc
    gc.collect()
    torch.cuda.empty_cache()
    if RELEASE_GPU_WHEN_DONE and IN_COLAB:
        from google.colab import runtime
        print("Releasing the Colab runtime.")
        runtime.unassign()


## 5. Review on CPU

Open notebook 20 to inspect completed pairs, interruption records, averages, and figures.
It downloads reports rather than models. Do not occupy a GPU for analysis.